## Setup: Create Sample Genie Spaces for Migration Testing

This notebook creates sample data and Genie Spaces in the source catalog for testing the migration workflow.

**What it creates:**
1. Sample healthcare/supply chain data tables
2. Three Genie Spaces with instructions, sample questions, and benchmarks
3. Verifies spaces are accessible

## Install Dependencies

In [ ]:
%pip install databricks-sdk>=0.76.0 --quiet
dbutils.library.restartPython()

## Configuration

In [ ]:
dbutils.widgets.text("catalog", "YOUR_SOURCE_CATALOG", "Catalog")
dbutils.widgets.text("schema", "genie_demo", "Schema")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")

print(f"Catalog: {CATALOG}")
print(f"Schema: {SCHEMA}")

## Create Schema

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"Schema {CATALOG}.{SCHEMA} ready")

## Create Sample Data Tables

Synthetic healthcare-style supply chain sample data for demo Genie Spaces.

In [ ]:
from pyspark.sql import Row
from datetime import date, timedelta
import random

random.seed(42)

regions = ["Northeast", "Southeast", "Midwest", "Southwest", "West"]
contract_types = ["GPO Contract", "Local Contract", "Spot Buy"]
product_categories = ["Medical Supplies", "Pharmaceuticals", "Equipment", "PPE", "Lab Supplies"]
suppliers = ["MedSupply Co", "PharmaCorp", "HealthEquip Inc", "SafetyFirst", "LabTech"]
hospitals = ["Memorial Hospital", "St. Mary's Medical", "University Health", "Community General", "Regional Medical Center"]
delivery_status = ["Delivered", "In Transit", "Backordered", "Pending"]

purchase_orders = []
for i in range(100):
    qty = random.randint(10, 500)
    unit_price = round(random.uniform(5, 200), 2)
    contract_price = round(unit_price * 0.85, 2)
    contract_type = random.choice(contract_types)
    total = round(qty * unit_price, 2)
    savings = round(qty * (unit_price - contract_price), 2) if contract_type == "GPO Contract" else 0.0
    
    purchase_orders.append(Row(
        order_id=f"PO-{2024000 + i}",
        order_date=date(2024, 1, 1) + timedelta(days=random.randint(0, 90)),
        member_hospital=random.choice(hospitals),
        region=random.choice(regions),
        supplier_name=random.choice(suppliers),
        product_category=random.choice(product_categories),
        quantity=qty,
        unit_price=unit_price,
        contract_price=contract_price,
        total_amount=total,
        savings=savings,
        contract_type=contract_type,
        delivery_status=random.choice(delivery_status)
    ))

po_df = spark.createDataFrame(purchase_orders)
po_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.purchase_orders")
print(f"Created {CATALOG}.{SCHEMA}.purchase_orders ({po_df.count()} rows)")

In [ ]:
supplier_data = [
    Row(supplier_name="MedSupply Co", category_specialty="Medical Supplies", compliance_rate=92.5, quality_score=4.5, contract_status="Active"),
    Row(supplier_name="PharmaCorp", category_specialty="Pharmaceuticals", compliance_rate=88.0, quality_score=4.2, contract_status="Active"),
    Row(supplier_name="HealthEquip Inc", category_specialty="Equipment", compliance_rate=95.0, quality_score=4.8, contract_status="Active"),
    Row(supplier_name="SafetyFirst", category_specialty="PPE", compliance_rate=78.5, quality_score=3.9, contract_status="Under Review"),
    Row(supplier_name="LabTech", category_specialty="Lab Supplies", compliance_rate=91.0, quality_score=4.4, contract_status="Active"),
]

supplier_df = spark.createDataFrame(supplier_data)
supplier_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.suppliers")
print(f"Created {CATALOG}.{SCHEMA}.suppliers ({supplier_df.count()} rows)")

In [ ]:
contract_data = [
    Row(contract_id="CNT-001", vendor="MedSupply Co", category="CDB", signed_date=date(2023, 1, 15), effective_start=date(2023, 2, 1), effective_end=date(2025, 1, 31), status="Active"),
    Row(contract_id="CNT-002", vendor="PharmaCorp", category="ODB", signed_date=date(2023, 3, 1), effective_start=date(2023, 4, 1), effective_end=date(2025, 3, 31), status="Active"),
    Row(contract_id="CNT-003", vendor="HealthEquip Inc", category="PA", signed_date=date(2023, 6, 15), effective_start=date(2023, 7, 1), effective_end=date(2024, 6, 30), status="Expired"),
    Row(contract_id="CNT-004", vendor="SafetyFirst", category="CPSC", signed_date=date(2024, 1, 1), effective_start=date(2024, 1, 15), effective_end=date(2026, 1, 14), status="Active"),
    Row(contract_id="CNT-005", vendor="LabTech", category="CDB", signed_date=date(2023, 9, 1), effective_start=date(2023, 10, 1), effective_end=date(2025, 9, 30), status="Active"),
]

contract_df = spark.createDataFrame(contract_data)
contract_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.contracts")
print(f"Created {CATALOG}.{SCHEMA}.contracts ({contract_df.count()} rows)")

In [ ]:
product_data = [
    Row(product_code="MED-001", product_name="Surgical Gloves", category="Medical Supplies", unit_cost=0.50, gpo_price=0.42),
    Row(product_code="MED-002", product_name="Bandages", category="Medical Supplies", unit_cost=2.00, gpo_price=1.70),
    Row(product_code="PHR-001", product_name="Pain Relief", category="Pharmaceuticals", unit_cost=15.00, gpo_price=12.50),
    Row(product_code="PHR-002", product_name="Antibiotics", category="Pharmaceuticals", unit_cost=25.00, gpo_price=21.00),
    Row(product_code="EQP-001", product_name="Blood Pressure Monitor", category="Equipment", unit_cost=150.00, gpo_price=125.00),
    Row(product_code="PPE-001", product_name="N95 Masks", category="PPE", unit_cost=3.00, gpo_price=2.50),
    Row(product_code="LAB-001", product_name="Test Tubes", category="Lab Supplies", unit_cost=0.25, gpo_price=0.20),
]

product_df = spark.createDataFrame(product_data)
product_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.products")
print(f"Created {CATALOG}.{SCHEMA}.products ({product_df.count()} rows)")

In [ ]:
facility_data = [
    Row(facility_id="FAC-001", facility_name="Memorial Hospital", region="Northeast", bed_count=500, tier="Large"),
    Row(facility_id="FAC-002", facility_name="St. Mary's Medical", region="Southeast", bed_count=350, tier="Medium"),
    Row(facility_id="FAC-003", facility_name="University Health", region="Midwest", bed_count=800, tier="Large"),
    Row(facility_id="FAC-004", facility_name="Community General", region="Southwest", bed_count=200, tier="Small"),
    Row(facility_id="FAC-005", facility_name="Regional Medical Center", region="West", bed_count=450, tier="Medium"),
]

facility_df = spark.createDataFrame(facility_data)
facility_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.facilities")
print(f"Created {CATALOG}.{SCHEMA}.facilities ({facility_df.count()} rows)")

## Connect to Workspace and Find Warehouse

In [ ]:
from databricks.sdk import WorkspaceClient
import json

w = WorkspaceClient()
host = w.config.host
print(f"Connected to: {host}")

warehouse_id = None
for wh in w.warehouses.list():
    state = str(wh.state).upper() if wh.state else ""
    if "RUNNING" in state:
        warehouse_id = wh.id
        print(f"Using warehouse: {wh.name} ({wh.id}) - {state}")
        break

if not warehouse_id:
    warehouses = list(w.warehouses.list())
    if warehouses:
        warehouse_id = warehouses[0].id
        print(f"Using warehouse: {warehouses[0].name} ({warehouse_id}) - will start if needed")
    else:
        raise Exception("No SQL warehouse found")

## Helper Function: Create or Update Genie Space

Uses version 1 format with sorted 32-char hex IDs (required by Genie API).

In [ ]:
def gen_sorted_id(index):
    """Generate a 32-char hex ID that sorts correctly (required by Genie API)."""
    return f"{index:032x}"

def create_or_update_genie_space(client, name, description, tables, instructions, benchmarks=None):
    """
    Create or update a Genie Space using version 1 format.
    
    Note: The Genie API requires:
    - IDs must be 32-character lowercase hex (UUID without hyphens)
    - example_question_sqls must be sorted by ID
    
    Returns: space_id
    """
    serialized = {
        "version": 1,
        "data_sources": {
            "tables": [{"identifier": t} for t in tables]
        },
        "instructions": {
            "text_instructions": [
                {
                    "id": gen_sorted_id(0),
                    "content": [instructions.strip()]
                }
            ],
            "example_question_sqls": [
                {
                    "id": gen_sorted_id(i + 1),
                    "question": [bm["question"]],
                    "sql": [bm["sql"]]
                }
                for i, bm in enumerate(benchmarks or [])
            ]
        }
    }
    serialized_json = json.dumps(serialized)
    
    existing = None
    try:
        resp = client.genie.list_spaces()
        for space in resp.spaces or []:
            if getattr(space, "title", None) == name:
                existing = space
                break
    except Exception as e:
        print(f"  Could not list spaces: {e}")
    
    if existing:
        try:
            client.genie.update_space(
                space_id=existing.space_id,
                title=name,
                description=description,
                warehouse_id=warehouse_id,
                serialized_space=serialized_json
            )
            print(f"  Updated: {existing.space_id}")
            return existing.space_id
        except Exception as e:
            print(f"  Update failed: {e}")
            return existing.space_id
    else:
        try:
            result = client.genie.create_space(
                title=name,
                description=description,
                warehouse_id=warehouse_id,
                serialized_space=serialized_json
            )
            print(f"  Created: {result.space_id}")
            return result.space_id
        except Exception as e:
            print(f"  Create failed: {e}")
            return None

## Create Genie Space 1: Supply Chain Analytics

In [ ]:
SPACE1_NAME = "Supply Chain Analytics"
SPACE1_DESC = "Analyze purchase orders, supplier performance, contract compliance, and savings opportunities."

SPACE1_TABLES = [
    f"{CATALOG}.{SCHEMA}.purchase_orders",
    f"{CATALOG}.{SCHEMA}.suppliers",
]

SPACE1_INSTRUCTIONS = f"""
All monetary values should be formatted with dollar signs and commas.
When asked about "savings", use the savings column from purchase_orders.
"Off-contract spend" means rows where contract_type is NOT 'GPO Contract'.
"Contract leakage" is synonymous with "off-contract spend".
When comparing suppliers, always include compliance_rate and quality_score.
Default to showing top 10 results unless specified.
Round percentages to 1 decimal place.
GPO = Group Purchasing Organization.
Join purchase_orders to suppliers on supplier_name.
"""

SPACE1_BENCHMARKS = [
    {
        "question": "What is total spend by region?",
        "sql": f"SELECT region, SUM(total_amount) as total_spend FROM {CATALOG}.{SCHEMA}.purchase_orders GROUP BY region ORDER BY total_spend DESC"
    },
    {
        "question": "Which suppliers have low compliance?",
        "sql": f"SELECT supplier_name, compliance_rate FROM {CATALOG}.{SCHEMA}.suppliers WHERE compliance_rate < 90 ORDER BY compliance_rate"
    },
    {
        "question": "What are total GPO savings?",
        "sql": f"SELECT SUM(savings) as total_savings FROM {CATALOG}.{SCHEMA}.purchase_orders WHERE contract_type = 'GPO Contract'"
    }
]

print(f"Creating: {SPACE1_NAME}")
space1_id = create_or_update_genie_space(w, SPACE1_NAME, SPACE1_DESC, SPACE1_TABLES, SPACE1_INSTRUCTIONS, SPACE1_BENCHMARKS)

## Create Genie Space 2: Contract Intelligence

In [ ]:
SPACE2_NAME = "Contract Intelligence"
SPACE2_DESC = "Analyze contracts, vendors, and products. Identify expiring contracts and pricing opportunities."

SPACE2_TABLES = [
    f"{CATALOG}.{SCHEMA}.contracts",
    f"{CATALOG}.{SCHEMA}.products",
]

SPACE2_INSTRUCTIONS = f"""
Contract categories: CDB (Clinical Distribution Business), ODB (Office-Based Distribution), PA (Physician Advocacy), CPSC (Clinical Pharmaceutical Supply Chain).
Contract status: Active, Expired, Under Review.
When analyzing pricing, compare unit_cost to gpo_price to show savings potential.
"Expiring contracts" means contracts where effective_end is within 90 days of today.
Format dates as YYYY-MM-DD.
"""

SPACE2_BENCHMARKS = [
    {
        "question": "How many active contracts?",
        "sql": f"SELECT COUNT(*) as active_contracts FROM {CATALOG}.{SCHEMA}.contracts WHERE status = 'Active'"
    },
    {
        "question": "Show contracts by category",
        "sql": f"SELECT category, COUNT(*) as contract_count FROM {CATALOG}.{SCHEMA}.contracts GROUP BY category ORDER BY contract_count DESC"
    }
]

print(f"Creating: {SPACE2_NAME}")
space2_id = create_or_update_genie_space(w, SPACE2_NAME, SPACE2_DESC, SPACE2_TABLES, SPACE2_INSTRUCTIONS, SPACE2_BENCHMARKS)

## Create Genie Space 3: Facility Insights

In [ ]:
SPACE3_NAME = "Facility Insights"
SPACE3_DESC = "Analyze healthcare facility data including regional distribution and capacity."

SPACE3_TABLES = [
    f"{CATALOG}.{SCHEMA}.facilities",
    f"{CATALOG}.{SCHEMA}.purchase_orders",
]

SPACE3_INSTRUCTIONS = f"""
Facility tiers: Large (500+ beds), Medium (200-499 beds), Small (<200 beds).
Regions: Northeast, Southeast, Midwest, Southwest, West.
Join facilities to purchase_orders on facility_name = member_hospital for spend analysis.
Show bed counts when comparing facilities.
"""

SPACE3_BENCHMARKS = []

print(f"Creating: {SPACE3_NAME}")
space3_id = create_or_update_genie_space(w, SPACE3_NAME, SPACE3_DESC, SPACE3_TABLES, SPACE3_INSTRUCTIONS, SPACE3_BENCHMARKS)

## Verify Genie Spaces

In [ ]:
print("\nVerifying Genie Spaces...\n")

created_spaces = []

for space_name, space_id in [
    (SPACE1_NAME, space1_id),
    (SPACE2_NAME, space2_id),
    (SPACE3_NAME, space3_id)
]:
    if space_id:
        try:
            space = w.genie.get_space(space_id=space_id, include_serialized_space=True)
            bm_count = 0
            if space.serialized_space:
                data = json.loads(space.serialized_space)
                bm_count = len(data.get("instructions", {}).get("example_question_sqls", []))
            
            print(f"PASS {space_name}")
            print(f"  ID: {space_id}")
            print(f"  Benchmarks: {bm_count}")
            print(f"  URL: {host}/genie/rooms/{space_id}")
            print()
            created_spaces.append({"name": space_name, "id": space_id, "benchmarks": bm_count})
        except Exception as e:
            print(f"FAIL {space_name}: Could not verify - {e}")
    else:
        print(f"FAIL {space_name}: Not created")

## Summary

In [ ]:
print("=" * 70)
print("SETUP COMPLETE")
print("=" * 70)
print(f"\nCatalog: {CATALOG}")
print(f"Schema:  {SCHEMA}")
print(f"\nTables created:")
print(f"  - {CATALOG}.{SCHEMA}.purchase_orders (100 rows)")
print(f"  - {CATALOG}.{SCHEMA}.suppliers (5 rows)")
print(f"  - {CATALOG}.{SCHEMA}.contracts (5 rows)")
print(f"  - {CATALOG}.{SCHEMA}.products (7 rows)")
print(f"  - {CATALOG}.{SCHEMA}.facilities (5 rows)")
print(f"\nGenie Spaces created: {len(created_spaces)}")
for s in created_spaces:
    print(f"  - {s['name']} ({s['benchmarks']} benchmarks)")
print("\n" + "=" * 70)
print("NEXT STEPS:")
print("=" * 70)
print("1. Run src_genie_inventory to discover these spaces")
print("2. Open Src_02_Review_and_Approve and set confirmation=CONFIRM")
print("3. Run src_genie_export to export them")
print("4. Run tgt_genie_deploy to migrate to target catalog")
print("\nOr test the spaces directly:")
for s in created_spaces:
    print(f"  {host}/genie/rooms/{s['id']}")